# Principal Component Analysis (PCA) - Detailed Guide

**Understanding dimensionality reduction through PCA with hands-on implementation and visualization.**

## Project Overview

This project provides a comprehensive walkthrough of Principal Component Analysis (PCA). We use the Iris dataset to demonstrate how PCA works, when to use it, and how to interpret its results. PCA is the most widely used dimensionality reduction technique in data science.

## Learning Objectives

1. Understand the mathematical intuition behind PCA
2. Implement PCA step-by-step from scratch and with sklearn
3. Interpret explained variance ratios and choose the number of components
4. Visualize high-dimensional data in 2D and 3D
5. Understand when PCA is appropriate and when it fails

## Business / Research Problem

High-dimensional data is common in genomics, image processing, NLP, and sensor data. PCA helps:
- **Reduce computational cost** by lowering feature dimensions
- **Visualize clusters** in 2D/3D from many features
- **Remove noise** by discarding low-variance components
- **Address multicollinearity** in regression

**Key question:** *How can we reduce the dimensionality of data while preserving maximum information?*

## Why This Analysis Matters

The curse of dimensionality affects almost every ML pipeline. PCA is essential for preprocessing, visualization, and feature engineering in high-dimensional settings.

## Dataset Overview

We use the classic **Iris dataset** (150 samples, 4 features, 3 classes) — perfect for demonstrating PCA because:
- Small enough to understand fully
- 4D data can be reduced to 2D with good class separation
- Well-known benchmark for visualization

## Dataset Source & License Notes

- **Source:** sklearn built-in dataset (Fisher's Iris dataset)
- **License:** Public domain
- **Usage:** No download needed — built into scikit-learn

## Environment Setup

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy scikit-learn -q

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
print('All imports loaded successfully.')

## Configuration / Constants

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## Dataset Download / Loading

In [ ]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)
print(f'Loaded {len(df)} samples with {len(iris.feature_names)} features')
print(f'Classes: {list(iris.target_names)}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Shape: {df.shape}')
print(f'Missing: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'\nClass distribution:\n{df["species"].value_counts()}')
df.describe()

## Data Cleaning

The Iris dataset is clean, but PCA requires standardization since it's variance-sensitive.

In [ ]:
features = iris.feature_names
X = df[features].values

# Standardize features (CRITICAL for PCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Before scaling:')
print(f'  Means: {X.mean(axis=0).round(2)}')
print(f'  Stds:  {X.std(axis=0).round(2)}')
print('\nAfter scaling:')
print(f'  Means: {X_scaled.mean(axis=0).round(6)}')
print(f'  Stds:  {X_scaled.std(axis=0).round(2)}')

## Exploratory Data Analysis

In [ ]:
# Pairplot to see original feature relationships
g = sns.pairplot(df, hue='species', vars=features, diag_kind='kde',
                 plot_kws={'alpha': 0.6, 's': 20})
g.figure.suptitle('Original Feature Space (4D)', y=1.02)
plt.show()

## Univariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, feat in enumerate(features):
    ax = axes[i // 2, i % 2]
    for sp in iris.target_names:
        subset = df[df['species'] == sp][feat]
        ax.hist(subset, bins=15, alpha=0.5, label=sp)
    ax.set_title(feat)
    ax.legend()
plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

### Step-by-Step PCA

PCA works by:
1. Standardizing the data
2. Computing the covariance matrix
3. Finding eigenvalues and eigenvectors
4. Projecting data onto top eigenvectors

In [ ]:
# Step 1: Covariance matrix
cov_matrix = np.cov(X_scaled.T)
print('Covariance Matrix:')
print(pd.DataFrame(cov_matrix, columns=features, index=features).round(3))

# Step 2: Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
print(f'\nEigenvalues: {eigenvalues.round(4)}')
print(f'\nExplained variance ratio: {(eigenvalues / eigenvalues.sum()).round(4)}')

In [ ]:
# Using sklearn PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

print('=== PCA Results ===')
for i, (var, cum) in enumerate(zip(pca.explained_variance_ratio_,
                                    np.cumsum(pca.explained_variance_ratio_))):
    print(f'PC{i+1}: {var:.4f} ({var*100:.1f}%) | Cumulative: {cum:.4f} ({cum*100:.1f}%)')

## Feature-Specific Insights

### Explained Variance - The Scree Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, 5), pca.explained_variance_ratio_, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')
axes[0].set_xticks(range(1, 5))

# Cumulative variance
cum_var = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(1, 5), cum_var, 'bo-', linewidth=2)
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
axes[1].fill_between(range(1, 5), cum_var, alpha=0.2)
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xticks(range(1, 5))
axes[1].legend()

plt.tight_layout()
plt.show()

n_95 = np.argmax(cum_var >= 0.95) + 1
print(f'Components needed for 95% variance: {n_95}')

In [ ]:
# Component loadings
loadings = pd.DataFrame(pca.components_.T, columns=[f'PC{i+1}' for i in range(4)],
                         index=features)
print('=== Component Loadings ===')
print(loadings.round(4))
print('\nHigh absolute loading = feature contributes strongly to that PC')

## Statistical Checks / Hypothesis Testing

In [ ]:
# Verify PCA preserves total variance
original_var = X_scaled.var(axis=0).sum()
pca_var = pca.explained_variance_.sum()
print(f'Original total variance: {original_var:.4f}')
print(f'PCA total variance: {pca_var:.4f}')
print(f'Variance preserved: {pca_var/original_var*100:.2f}%')

# PC components are orthogonal (uncorrelated)
pc_corr = np.corrcoef(X_pca.T)
print(f'\nPC correlation matrix (should be identity):')
print(np.round(pc_corr, 6))

## Visual Analysis

### 2D PCA Projection — The Main Payoff

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
for i, (species, color) in enumerate(zip(iris.target_names, colors)):
    mask = iris.target == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=species,
               alpha=0.7, s=60, edgecolors='white', linewidth=0.5)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('PCA: 4D Iris Data Projected to 2D')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Biplot: show both data points and feature vectors
fig, ax = plt.subplots(figsize=(10, 8))

for i, (species, color) in enumerate(zip(iris.target_names, colors)):
    mask = iris.target == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=species, alpha=0.5, s=40)

# Feature vectors
scale = 3
for i, feat in enumerate(features):
    ax.arrow(0, 0, pca.components_[0, i] * scale, pca.components_[1, i] * scale,
             head_width=0.1, head_length=0.05, fc='black', ec='black')
    ax.text(pca.components_[0, i] * scale * 1.15, pca.components_[1, i] * scale * 1.15,
            feat, fontsize=9, ha='center')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA Biplot')
ax.legend()
plt.tight_layout()
plt.show()

## Key Findings

1. **PC1 and PC2 capture ~95% of variance** — 4D can be effectively reduced to 2D
2. **PC1 separates setosa** from versicolor/virginica
3. **PC2 partially separates versicolor and virginica**
4. **Petal features dominate PC1** — they carry the most discriminative information
5. **PCA components are orthogonal** — they capture independent patterns

## Limitations

1. **Linear only:** PCA assumes linear relationships — curved manifolds need kernel PCA or t-SNE
2. **Variance-based:** PCA maximizes variance, not class separation (use LDA for that)
3. **Sensitive to scaling:** Always standardize before PCA
4. **Interpretability:** PCs are linear combinations — harder to interpret than original features
5. **Outlier sensitive:** Extreme values can dominate principal components

## Recommendations / Next Steps

1. Compare PCA with t-SNE and UMAP for visualization
2. Use PCA as preprocessing for classification (compare accuracy with/without)
3. Explore kernel PCA for non-linear relationships

### How to Extend This Analysis
- Apply PCA to a high-dimensional dataset (MNIST, gene expression)
- Implement PCA from scratch using SVD
- Compare PCA with autoencoders for dimensionality reduction

## Common Mistakes

1. **Not scaling features:** PCA is variance-sensitive — unscaled features with large ranges dominate
2. **Choosing components arbitrarily:** Use the scree plot or 95% variance threshold
3. **Expecting class separation:** PCA maximizes variance, not discriminability — setosa separating is lucky
4. **Applying PCA to already low-dimensional data:** Limited benefit with <5 features
5. **Interpreting PCs as original features:** Each PC is a mix of all original features

## Mini Challenge / Exercises

1. What happens if you don't standardize the data? Run PCA on unscaled Iris data and compare
2. Apply PCA to the breast cancer dataset (30 features). How many PCs capture 95% variance?
3. Implement PCA from scratch using only numpy (eigendecomposition of the covariance matrix)
4. Compare the PCA 2D plot with a t-SNE 2D plot. Which shows better class separation?
5. Use PCA components as features in a classifier. Does accuracy change compared to original features?

In [ ]:
# Space for exercise solutions

## Final Summary / Key Takeaways

- **PCA finds the directions of maximum variance** in your data
- **Always standardize features first** — this is non-negotiable
- **The scree plot and cumulative variance** guide component selection
- **PCA preserves global structure** but may lose local details (unlike t-SNE)
- **Component loadings reveal which features matter most** for each PC
- **PCA is unsupervised** — it doesn't use labels, so class separation is a bonus, not a guarantee

PCA is a fundamental tool in every data scientist's toolkit — master it early.